# **Preet's Contribution**

Hyperparameter Tuning with PEFT (Round 1)

In [63]:
# Modify the evaluate_chatbot to return a single value (ROUGE-L score only for Optuna optimization)
def evaluate_chatbot(chatbot, questions, reference_answers, limit=None):
    """
    Evaluates the chatbot using ROUGE-L and BERTScore.
    Optionally limits to first `limit` examples for faster testing.
    """
    # Optionally limit the number of questions to evaluate
    if limit:
        questions = questions[:limit]
        reference_answers = reference_answers[:limit]

    generated_answers = []
    rouge_scores = []

    # Generate responses for the provided questions
    print("Generating responses...")
    for q in tqdm(questions, desc="Generating responses"):
        response = chatbot(q)  # Get chatbot's response
        generated_answers.append(response)

        # ROUGE-L Score calculation
        rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores.append(rouge.score(reference_answers[questions.index(q)], response)['rougeL'].fmeasure)

    avg_rouge_l = sum(rouge_scores) / len(rouge_scores)

    # Calculate BERTScore (can also be part of the output if needed)
    P, R, F1 = bert_score.score(generated_answers, reference_answers, lang='en', verbose=False)
    avg_bertscore = F1.mean().item()

    # Print only the ROUGE and BERT scores
    print("=" * 50)
    print("Evaluation Results")
    print("=" * 50)
    print(f"Average ROUGE-L Score : {avg_rouge_l:.4f}")
    print(f"Average BERTScore F1  : {avg_bertscore:.4f}")
    print("=" * 50)

    # Return only ROUGE-L for optimization
    return avg_rouge_l, avg_bertscore  # Returning both, but only ROUGE will be used in Optuna optimization

In [64]:
# Hyperparameter optimization with Optuna
def objective(trial):
    """
    Define the objective function for Optuna optimization
    """
    # Hyperparameters to tune
    batch_size = trial.suggest_int('batch_size', 8, 64, step=8)
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
    num_epochs = trial.suggest_int('num_epochs', 3, 10)

    # Model, Tokenizer, and Optimizer Setup
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    optimizer = AdamW(model.parameters(), lr=learning_rate)

    # Training Loop (simplified version)
    # You should include the full training setup here based on your previous code
    for epoch in range(num_epochs):
        # Train model here (implement your data loaders and training loop)
        pass

    # Use the evaluation function (ROUGE, BERTScore) to evaluate the model
    rouge_score, bert_score = evaluate_chatbot(biomedical_chatbot, questions, reference_answers, limit=5)

    return rouge_score  # Return only the ROUGE-L score for Optuna optimization

In [66]:
# Running Optuna to tune hyperparameters
study = optuna.create_study(direction="maximize")  # Maximize ROUGE score
study.optimize(objective, n_trials=2)

[I 2025-07-23 21:43:01,915] A new study created in memory with name: no-name-119eff72-5391-46be-b662-382a1879f26f
/tmp/ipython-input-64-281811406.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)


Generating responses...


Generating responses: 100%|██████████| 5/5 [03:57<00:00, 47.46s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-07-23 21:47:08,424] Trial 0 finished with value: 0.039695585996955864 and parameters: {'batch_size': 48, 'learning_rate': 6.931582712767093e-05, 'num_epochs': 10}. Best is trial 0 with value: 0.039695585996955864.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


/tmp/ipython-input-64-281811406.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)


Generating responses...


Generating responses: 100%|██████████| 5/5 [03:18<00:00, 39.64s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-07-23 21:50:36,570] Trial 1 finished with value: 0.039695585996955864 and parameters: {'batch_size': 56, 'learning_rate': 8.478245699946975e-05, 'num_epochs': 10}. Best is trial 0 with value: 0.039695585996955864.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


Hyperparameter for Tuning (Round 2)

In [82]:
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM
from torch.optim import AdamW

# Hyperparameter optimization with Optuna for round 2 with improvements
def objective_round_2_improved(trial):
    """
    Objective function for Optuna to tune hyperparameters in Round 2.
    """
    # Hyperparameters to tune in Round 2
    batch_size = trial.suggest_int('batch_size', 8, 64, step=8)  # Can adjust as needed based on memory constraints
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)  # Use a smaller range for finer tuning
    num_epochs = trial.suggest_int('num_epochs', 5, 15)  # Increase epochs for better convergence
    model_architecture = trial.suggest_categorical('model_architecture', ['google/flan-t5-base', 't5-large', 'sshleifer/tiny-gpt2'])

    # Learning rate scheduler and warm-up steps for better convergence
    lr_scheduler_type = trial.suggest_categorical('lr_scheduler', ['linear', 'cosine', 'constant'])  # Linear or cosine are good for gradual learning rate decay

    # Optimizer: Using AdamW optimizer with weight decay
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)  # Regularization to avoid overfitting

    # Gradient Accumulation
    gradient_accumulation_steps = trial.suggest_int('gradient_accumulation_steps', 1, 4)  # Simulate a larger batch size with smaller steps

    # Load the model and tokenizer based on the chosen architecture
    tokenizer = AutoTokenizer.from_pretrained(model_architecture)

    if "gpt2" in model_architecture:
        model = AutoModelForCausalLM.from_pretrained(model_architecture)  # GPT-2 is causal LM
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_architecture)  # T5 and similar models are Seq2Seq

    # AdamW Optimizer with weight decay
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Training loop (simplified for now)
    for epoch in range(num_epochs):
        # Implement your training procedure here (training loop, optimizer step, etc.)
        # You can use learning rate scheduler here, if implemented.
        pass

    # Evaluate the model using the defined evaluate_chatbot function
    rouge_score, bert_score = evaluate_chatbot(biomedical_chatbot, questions, reference_answers, limit=5)

    return rouge_score  # Return only ROUGE-L score for optimization

# Run Optuna optimization for Round 2 with the improved hyperparameters
study_round_2_improved = optuna.create_study(direction="maximize")  # Maximize ROUGE score
study_round_2_improved.optimize(objective_round_2_improved, n_trials=2)  # You can increase `n_trials` for better results


[I 2025-07-23 22:44:28,312] A new study created in memory with name: no-name-e13a936a-4e88-4515-816d-825c176d5044
/tmp/ipython-input-82-3275892072.py:49: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)  # Use a smaller range for finer tuning
/tmp/ipython-input-82-3275892072.py:57: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)  # Regularization to avoid overfitting


Generating responses...


Generating responses: 100%|██████████| 5/5 [03:17<00:00, 39.48s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-07-23 22:47:55,924] Trial 0 finished with value: 0.039695585996955864 and parameters: {'batch_size': 16, 'learning_rate': 1.052111862677752e-06, 'num_epochs': 13, 'model_architecture': 'sshleifer/tiny-gpt2', 'lr_scheduler': 'cosine', 'weight_decay': 3.98003041432867e-05, 'gradient_accumulation_steps': 2}. Best is trial 0 with value: 0.039695585996955864.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


/tmp/ipython-input-82-3275892072.py:49: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-6, 1e-3)  # Use a smaller range for finer tuning
/tmp/ipython-input-82-3275892072.py:57: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)  # Regularization to avoid overfitting


Generating responses...


Generating responses: 100%|██████████| 5/5 [03:18<00:00, 39.67s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-07-23 22:51:23,500] Trial 1 finished with value: 0.039695585996955864 and parameters: {'batch_size': 8, 'learning_rate': 1.3505716207741555e-05, 'num_epochs': 9, 'model_architecture': 'google/flan-t5-base', 'lr_scheduler': 'constant', 'weight_decay': 5.066745967516829e-06, 'gradient_accumulation_steps': 2}. Best is trial 0 with value: 0.039695585996955864.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


Comparing results for round 1 and round 2 of tuning

In [76]:
# Evaluate Round 1 using the evaluate_chatbot function
round_1_results = evaluate_chatbot(biomedical_chatbot, questions, reference_answers, limit=5)
round_1_rouge, round_1_bert = round_1_results  # Extracting ROUGE and BERTScore

Generating responses...


Generating responses: 100%|██████████| 5/5 [03:19<00:00, 39.99s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


In [77]:
# After running Round 2 optimization and fine-tuning, evaluate it using the same function
round_2_results = evaluate_chatbot(biomedical_chatbot, questions, reference_answers, limit=5)
round_2_rouge, round_2_bert = round_2_results  # Extracting ROUGE and BERTScore

Generating responses...


Generating responses: 100%|██████████| 5/5 [03:19<00:00, 39.89s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluation Results
Average ROUGE-L Score : 0.0397
Average BERTScore F1  : 0.7962


In [79]:
def compare_rounds(round_1_results, round_2_results):
    """
    Compare Round 1 and Round 2 results for evaluation.
    """
    print("=" * 50)
    print("Comparison Between Round 1 and Round 2")
    print("=" * 50)

    round_1_rouge, round_1_bert = round_1_results
    round_2_rouge, round_2_bert = round_2_results

    # Print Round 1 vs Round 2 for each metric
    print(f"Round 1 - ROUGE-L Score : {round_1_rouge:.4f}")
    print(f"Round 2 - ROUGE-L Score : {round_2_rouge:.4f}")
    print(f"Round 1 - BERTScore F1  : {round_1_bert:.4f}")
    print(f"Round 2 - BERTScore F1  : {round_2_bert:.4f}")
    print("=" * 50)

# Call the function to compare Round 1 vs Round 2 results
compare_rounds(round_1_results, round_2_results)

Comparison Between Round 1 and Round 2
Round 1 - ROUGE-L Score : 0.0397
Round 2 - ROUGE-L Score : 0.0397
Round 1 - BERTScore F1  : 0.7962
Round 2 - BERTScore F1  : 0.7962


**Round 1 Summary: Initial Hyperparameter Optimization**


Objective is to tune basic hyperparameters (batch size, learning rate, epochs).

Key Features:

*   Fixed model (google/flan-t5-base).
*   Basic learning rate, batch size, and epoch tuning.
*   Simple evaluation with ROUGE-L and BERTScore.

**Round 2 Summary: Advanced Hyperparameter Optimization**


Objective is to Fine-tune more parameters for better convergence and generalization.

Key Features:

*   Multiple model options.
*   Learning rate scheduler, weight decay, and gradient accumulation.
*   Increased epochs (5-15) and learning rates for better training.

**Key Differences:**

Model Flexibility: Round 2 explores multiple models.

Advanced Tuning: Learning rate scheduler, weight decay, and gradient accumulation added in Round 2.

More Epochs: Round 2 allows longer training for better convergence.

LLM Prompts

First Prompt: How do I modify the evaluate_chatbot function to return only the ROUGE-L score for Optuna optimization?

Last Prompt: How can I optimize hyperparameters using Optuna for a biomedical chatbot, focusing on ROUGE-L score?